# Ensemble -- LSTM + Linear Regression

Combines the LSTM and Linear Regression predictions via weighted sum (weights fit by SLSQP on a validation-only rollout, then frozen and applied once to the held-out test rollout). `build_ensemble.py` in this same folder is a newer, separate script version that also includes the Transformer.


In [1]:
"""GHI ensemble -- combines the LSTM and Linear Regression predictions via
weighted sum. Weights are fit by constrained least-squares (SLSQP,
minimizing pooled RMSE) on a validation-period rollout, then frozen and
applied once to the held-out test rollout.
"""
import json

import joblib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow import keras

np.random.seed(42)

I0000 00:00:1788284335.238594   34451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788284335.491662   34451 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788284336.816217   34451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ---------------------------------------------------------------------------
# 1. Load data + feature set -- identical to both families' notebooks.

In [3]:
# ---------------------------------------------------------------------------

DATA_PATH = '../../data/FYP_Cleaned_Dataset.csv'
WINDOW_SIZE = 24
OUTPUT_HORIZON = 24
TRAIN_FRAC = 0.7
VAL_FRAC = 0.1

df = pd.read_csv(DATA_PATH, parse_dates=['Datetime'])
df = df.sort_values(['State', 'Datetime']).reset_index(drop=True)
df['GHI_lag_1h'] = df.groupby('State')['GHI_W_m2'].shift(1)
df['GHI_lag_24h'] = df.groupby('State')['GHI_W_m2'].shift(24)
df = df.dropna(subset=['GHI_lag_1h', 'GHI_lag_24h']).reset_index(drop=True)

FEATURE_COLS = [
    'Latitude', 'Cloud_Coverage_Pct', 'Temp_Skin_C',
    'Relative_Humidity_Pct', 'Wind_Speed_10m_m_s_log',
    'Precipitable_Water_cm', 'Rainfall_mm_log', 'Is_Daylight',
    'Hour_sin', 'Hour_cos', 'Wind_Sine', 'Wind_Cosine',
    'GHI_lag_1h', 'GHI_lag_24h',
]
TARGET_COL = 'GHI_W_m2'


def split_state(group):
    n = len(group)
    train_end = int(n * TRAIN_FRAC)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))
    return group.iloc[:train_end], group.iloc[train_end:val_end], group.iloc[val_end:]


val_parts, test_parts = [], []
for state, group in df.groupby('State', sort=False):
    _, va, te = split_state(group)
    val_parts.append(va); test_parts.append(te)
val_df = pd.concat(val_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

VAL_START = val_df['Datetime'].min()
TEST_START = test_df['Datetime'].min()
print('Val period starts: ', VAL_START)
print('Test period starts:', TEST_START, '(validation-rollout origins are capped before this -- no leakage)')

Val period starts:  2022-12-31 23:00:00
Test period starts: 2024-01-01 00:00:00 (validation-rollout origins are capped before this -- no leakage)


In [4]:
# ---------------------------------------------------------------------------
# 2. Load the two best models + their own scalers.

In [5]:
# ---------------------------------------------------------------------------

MODELS = {
    'lstm': {
        'model': keras.models.load_model('../LSTM/BestModel (Tuned)/lstm_d_model.keras'),
        'x_scaler': joblib.load('../LSTM/BestModel (Tuned)/lstm_d_x_scaler.joblib'),
        'y_scaler': joblib.load('../LSTM/BestModel (Tuned)/lstm_d_y_scaler.joblib'),
        'kind': 'sequence',
        'val_rmse': 65.51359964374764,
        'run': 'lstm_d',
    },
    'linreg': {
        'model': joblib.load('../LinearRegression/BestModel (Tuned)/linreg_h_model.joblib'),
        'x_scaler': joblib.load('../LinearRegression/BestModel (Tuned)/linreg_h_x_scaler.joblib'),
        'y_scaler': joblib.load('../LinearRegression/BestModel (Tuned)/linreg_h_y_scaler.joblib'),
        'kind': 'flat',
        'val_rmse': 70.98520903923401,
        'run': 'linreg_h',
    },
}
NAMES = ['lstm', 'linreg']  # fixed column order for every P matrix below

inv = {k: 1.0 / v['val_rmse'] for k, v in MODELS.items()}
total_inv = sum(inv.values())
INV_RMSE_WEIGHTS = {k: v / total_inv for k, v in inv.items()}
print('Inverse-val-RMSE weights (heuristic, for comparison only):', INV_RMSE_WEIGHTS)


def predict_batch(name, raw_windows):
    """raw_windows: (n, WINDOW_SIZE, n_features) real units -> (n, OUTPUT_HORIZON) real units, clipped >=0."""
    spec = MODELS[name]
    x_scaler, y_scaler = spec['x_scaler'], spec['y_scaler']
    flat = raw_windows.reshape(-1, len(FEATURE_COLS))
    scaled = x_scaler.transform(flat).reshape(raw_windows.shape).astype('float32')

    if spec['kind'] == 'sequence':
        pred_scaled = spec['model'].predict(scaled, verbose=0)
    else:
        pred_scaled = spec['model'].predict(scaled.reshape(scaled.shape[0], -1))

    pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
    return np.clip(pred, 0, None)


def build_rollout_origins(start, end=None):
    """Midnight-origin D -> D+1 rollout windows, restricted to origins whose
    day D falls in [start, end). Uses the full continuous per-state timeline
    (has correct lag features), same construction as each model's own
    notebook, so predictions align origin-for-origin across both models."""
    state_arrays = {}
    for state, g in df.groupby('State', sort=False):
        g = g.sort_values('Datetime')
        g = g[g['Datetime'] >= start]
        if end is not None:
            g = g[g['Datetime'] < end]
        g = g.reset_index(drop=True)
        state_arrays[state] = {
            'dt': g['Datetime'].to_numpy(),
            'X': g[FEATURE_COLS].to_numpy(dtype='float32'),
            'y': g[TARGET_COL].to_numpy(dtype='float32'),
        }

    origin_windows, y_true_next_day, origin_meta = [], [], []
    for state, arr in state_arrays.items():
        dt, X, y = arr['dt'], arr['X'], arr['y']
        n = len(dt)
        midnight_idx = np.where(pd.DatetimeIndex(dt).hour == 0)[0]
        for p in midnight_idx:
            if p + 2 * WINDOW_SIZE > n:
                continue
            origin_windows.append(X[p:p + WINDOW_SIZE])
            y_true_next_day.append(y[p + WINDOW_SIZE:p + 2 * WINDOW_SIZE])
            origin_meta.append((state, dt[p]))

    return np.stack(origin_windows), np.stack(y_true_next_day), origin_meta


def pooled_metrics(y_true_flat, y_pred_flat):
    mae = mean_absolute_error(y_true_flat, y_pred_flat)
    rmse = np.sqrt(mean_squared_error(y_true_flat, y_pred_flat))
    r2 = r2_score(y_true_flat, y_pred_flat)
    mbe = float(np.mean(y_pred_flat - y_true_flat))
    nrmse = rmse / np.mean(y_true_flat) * 100
    return {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2), 'mbe': mbe, 'nrmse': float(nrmse)}

I0000 00:00:1788284340.321364   34451 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Inverse-val-RMSE weights (heuristic, for comparison only): {'lstm': 0.520042700182806, 'linreg': 0.47995729981719415}


In [6]:
# ---------------------------------------------------------------------------
# 3. Validation-period rollout (weight fitting only -- never touches test).

In [7]:
# ---------------------------------------------------------------------------

val_origin_windows, val_y_true, val_origin_meta = build_rollout_origins(VAL_START, TEST_START)
print(f'\n{len(val_origin_meta)} VALIDATION rollout origins (for weight fitting)')

val_preds = {name: predict_batch(name, val_origin_windows) for name in NAMES}
y_val_flat = val_y_true.flatten()
P_val = np.column_stack([val_preds[name].flatten() for name in NAMES])  # (N, 2): lstm, linreg


def rmse_objective(weights):
    pred = P_val @ weights
    return np.sqrt(mean_squared_error(y_val_flat, pred))


opt_result = minimize(
    rmse_objective,
    x0=np.array([0.5, 0.5]),
    method='SLSQP',
    bounds=[(0, 1)] * 2,
    constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1},
)
OPT_WEIGHTS = dict(zip(NAMES, opt_result.x))
print('Optimized weights (fit on validation rollout, minimizing pooled RMSE):', OPT_WEIGHTS)
print(f'  converged: {opt_result.success}, val RMSE at optimum: {opt_result.fun:.4f}')
print(f'  val RMSE at equal weights (0.5,0.5):          {rmse_objective(np.array([0.5, 0.5])):.4f}')
print(f'  val RMSE at inverse-RMSE heuristic weights:  '
      f'{rmse_objective(np.array([INV_RMSE_WEIGHTS[n] for n in NAMES])):.4f}')
for name in NAMES:
    onehot = np.zeros(2); onehot[NAMES.index(name)] = 1.0
    print(f'  val RMSE at {name} alone (weight=1):          {rmse_objective(onehot):.4f}')


5460 VALIDATION rollout origins (for weight fitting)


/home/kelocker/miniconda3/envs/fyp/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
I0000 00:00:1788284341.475705   34568 cuda_dnn.cc:461] Loaded cuDNN version 92400


Optimized weights (fit on validation rollout, minimizing pooled RMSE): {'lstm': np.float64(0.8523119893538607), 'linreg': np.float64(0.14768801064613932)}
  converged: True, val RMSE at optimum: 68.8220
  val RMSE at equal weights (0.5,0.5):          69.6749
  val RMSE at inverse-RMSE heuristic weights:  69.5812
  val RMSE at lstm alone (weight=1):          68.9727
  val RMSE at linreg alone (weight=1):          73.6737


/home/kelocker/miniconda3/envs/fyp/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [8]:
# ---------------------------------------------------------------------------
# 4. Test rollout -- frozen weights applied ONCE, held-out data.

In [9]:
# ---------------------------------------------------------------------------

test_origin_windows, test_y_true, test_origin_meta = build_rollout_origins(TEST_START)
print(f'\n{len(test_origin_meta)} TEST rollout origins (final, held-out evaluation)')

test_preds = {name: predict_batch(name, test_origin_windows) for name in NAMES}
y_test_flat = test_y_true.flatten()
P_test = np.column_stack([test_preds[name].flatten() for name in NAMES])

pred_mean = P_test.mean(axis=1)
pred_inv_rmse = P_test @ np.array([INV_RMSE_WEIGHTS[n] for n in NAMES])
pred_optimized = P_test @ opt_result.x

metrics_mean = pooled_metrics(y_test_flat, pred_mean)
metrics_inv_rmse = pooled_metrics(y_test_flat, pred_inv_rmse)
metrics_optimized = pooled_metrics(y_test_flat, pred_optimized)

print('\n--- sanity check: individual model TEST-rollout metrics (should match summary_*.json) ---')
for name in NAMES:
    print(f'  {name:12s}', pooled_metrics(y_test_flat, test_preds[name].flatten()))

print('\n--- ensemble TEST-rollout metrics (weights frozen from validation, applied once) ---')
print('  simple mean:          ', metrics_mean)
print('  inverse-RMSE weighted:', metrics_inv_rmse)
print('  optimized (SLSQP):    ', metrics_optimized)

lstm_alone_rmse = pooled_metrics(y_test_flat, test_preds['lstm'].flatten())['rmse']
print(f'\nLSTM alone test-rollout RMSE: {lstm_alone_rmse:.4f}')
print(f'Optimized ensemble beats LSTM alone: {metrics_optimized["rmse"] < lstm_alone_rmse}')


10935 TEST rollout origins (final, held-out evaluation)


/home/kelocker/miniconda3/envs/fyp/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(



--- sanity check: individual model TEST-rollout metrics (should match summary_*.json) ---
  lstm         {'mae': 35.058589935302734, 'rmse': 70.63131539794335, 'r2': 0.933732271194458, 'mbe': 3.5419342517852783, 'nrmse': 34.78697121723309}
  linreg       {'mae': 41.396873474121094, 'rmse': 74.77616664189365, 'r2': 0.9257265329360962, 'mbe': 6.739506244659424, 'nrmse': 36.82837197708939}

--- ensemble TEST-rollout metrics (weights frozen from validation, applied once) ---
  simple mean:           {'mae': 37.383522033691406, 'rmse': 71.00521960347352, 'r2': 0.9330288171768188, 'mbe': 5.140720367431641, 'nrmse': 34.971124588333396}
  inverse-RMSE weighted: {'mae': 37.25617325021117, 'rmse': 70.92292185879695, 'r2': 0.9331839620618119, 'mbe': 5.076632146863097, 'nrmse': 34.930591727531116}
  optimized (SLSQP):     {'mae': 35.54707228525669, 'rmse': 70.37562640426148, 'r2': 0.9342111904710636, 'mbe': 4.014177317764329, 'nrmse': 34.6610405926417}

LSTM alone test-rollout RMSE: 70.6313
Optim

/home/kelocker/miniconda3/envs/fyp/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [10]:
# ---------------------------------------------------------------------------
# 5. Alignment sanity check -- one dataframe, eyeballed + NaN-checked, before
#    trusting any number above.

In [11]:
# ---------------------------------------------------------------------------

n_show_origins = 3
rows = []
for i in range(n_show_origins):
    state, origin_dt = test_origin_meta[i]
    for h in range(OUTPUT_HORIZON):
        rows.append({
            'state': state,
            'origin_dt': origin_dt,
            'horizon_h': h + 1,
            'actual': test_y_true[i, h],
            'lstm_pred': test_preds['lstm'][i, h],
            'linreg_pred': test_preds['linreg'][i, h],
        })
comparison = pd.DataFrame(rows)
print('\n--- alignment sanity check (first 3 test-rollout origins, all 24 horizons) ---')
print(comparison.head(20).to_string(index=False))
print('\nNaN check across full test-rollout arrays:')
print('  y_true_next_day NaNs:', np.isnan(test_y_true).sum())
for name in NAMES:
    print(f'  {name} pred NaNs:', np.isnan(test_preds[name]).sum())
print(f'shape check -- all equal to {test_y_true.shape}:',
      all(test_preds[n].shape == test_y_true.shape for n in NAMES))


--- alignment sanity check (first 3 test-rollout origins, all 24 horizons) ---
              state  origin_dt  horizon_h     actual  lstm_pred  linreg_pred
Johor (Johor Bahru) 2024-01-01          1   0.000000   7.426621     0.000000
Johor (Johor Bahru) 2024-01-01          2   0.000000   0.000000     0.000000
Johor (Johor Bahru) 2024-01-01          3   0.000000   0.000000     0.000000
Johor (Johor Bahru) 2024-01-01          4   0.000000   3.916259     0.000000
Johor (Johor Bahru) 2024-01-01          5   0.000000   0.000000     8.038116
Johor (Johor Bahru) 2024-01-01          6   0.000000   0.000000    31.639580
Johor (Johor Bahru) 2024-01-01          7  25.920000  25.332119    66.089951
Johor (Johor Bahru) 2024-01-01          8 168.649994 134.046036   159.542709
Johor (Johor Bahru) 2024-01-01          9 353.850006 279.590210   284.314270
Johor (Johor Bahru) 2024-01-01         10 489.899994 415.677307   410.619141
Johor (Johor Bahru) 2024-01-01         11 654.080017 509.705383   488.073

In [12]:
# ---------------------------------------------------------------------------
# 6. Per-horizon / per-state breakdown for the optimized-weight ensemble.

In [13]:
# ---------------------------------------------------------------------------

y_pred_next_day = pred_optimized.reshape(test_y_true.shape)

horizon_rows = []
for h in range(WINDOW_SIZE):
    yt, yp = test_y_true[:, h], y_pred_next_day[:, h]
    horizon_rows.append({
        'horizon_h': h + 1,
        'mae': mean_absolute_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp),
    })
horizon_df = pd.DataFrame(horizon_rows)

state_of_origin = np.array([m[0] for m in test_origin_meta])
per_state_rows = []
for state in sorted(set(state_of_origin)):
    mask = state_of_origin == state
    yt = test_y_true[mask].flatten()
    yp = y_pred_next_day[mask].flatten()
    per_state_rows.append({
        'state': state, 'n_origins': int(mask.sum()),
        'mae': mean_absolute_error(yt, yp),
        'rmse': np.sqrt(mean_squared_error(yt, yp)),
        'r2': r2_score(yt, yp),
    })
per_state_df = pd.DataFrame(per_state_rows)

horizon_df.to_csv('rolling_origin_direct_by_horizon.csv', index=False)
per_state_df.to_csv('rolling_origin_direct_by_state.csv', index=False)
print('\nsaved rolling_origin_direct_by_horizon.csv, rolling_origin_direct_by_state.csv (optimized-weight ensemble)')

summary = {
    'run_name': 'ensemble',
    'members': {name: {'run': spec['run'], 'val_rmse': spec['val_rmse']} for name, spec in MODELS.items()},
    'weight_fitting': {
        'method': 'SLSQP, minimize pooled RMSE on VALIDATION rollout (24h, midnight-origin), '
                  'weights in [0,1] summing to 1; frozen and applied once to the held-out test rollout.',
        'n_validation_rollout_origins': len(val_origin_meta),
        'converged': bool(opt_result.success),
        'validation_rmse_at_optimum': float(opt_result.fun),
        'optimized_weights': {n: float(w) for n, w in OPT_WEIGHTS.items()},
        'inverse_rmse_heuristic_weights': INV_RMSE_WEIGHTS,
    },
    'n_test_rollout_origins': len(test_origin_meta),
    'test_rollout_pooled': {
        'mean': metrics_mean,
        'inverse_rmse_weighted': metrics_inv_rmse,
        'optimized': metrics_optimized,
    },
    'per_model_test_rollout_pooled': {name: pooled_metrics(y_test_flat, test_preds[name].flatten()) for name in NAMES},
    'lstm_alone_beats_optimized_ensemble': bool(lstm_alone_rmse < metrics_optimized['rmse']),
}
with open('summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('saved summary.json')


saved rolling_origin_direct_by_horizon.csv, rolling_origin_direct_by_state.csv (optimized-weight ensemble)
saved summary.json
